In [2]:
%%capture
!pip install unsloth

### Global variables


In [4]:
dataset_id = (
    input("enter your huggingface dataset_id here, : ")
    or "pauliusztin/second_brain_course_summarization_task"
)
print(f"{dataset_id=}")

model_id = ( 
    input("enter your huggingface smmarization model id here, :")
    or "Parth19Dhimmar/Meta-Llama-3.1-8B-Instruct-second-brain"
)
print(f"{model_id=}")

dataset_id='pauliusztin/second_brain_course_summarization_task'
model_id='Parth19Dhimmar/Meta-Llama-3.1-8B-Instruct-second-brain'


In [5]:
from datasets import load_dataset

dataset = load_dataset(dataset_id, split="test")

In [11]:
dataset[26]["instruction"]

'[![Hugging Face\'s logo](/front/assets/huggingface_logo-noborder.svg) Hugging Face](/)\n\n  * [ Models](/models)\n  * [ Datasets](/datasets)\n  * [ Spaces](/spaces)\n  * [ Posts](/posts)\n  * [ Docs](/docs)\n  * [ Enterprise](/enterprise)\n  * [Pricing](/pricing)\n  * [Log In](/login)\n  * [Sign Up](/join)\n\n\n\nAmazon SageMaker documentation\n\nDeploy models to Amazon SageMaker\n\n# Amazon SageMaker\n\n🏡 View all docsAWS Trainium & InferentiaAccelerateAmazon SageMakerArgillaAutoTrainBitsandbytesChat UICompetitionsDataset viewerDatasetsDiffusersDistilabelEvaluateGoogle CloudGoogle TPUsGradioHubHub Python LibraryHugging Face Generative AI Services (HUGS)Huggingface.jsInference API (serverless)Inference Endpoints (dedicated)LeaderboardsLightevalOptimumPEFTSafetensorsSentence TransformersTRLTasksText Embeddings InferenceText Generation InferenceTokenizersTransformersTransformers.jssmolagentstimm\n\nSearch documentation\n\n`⌘K`\n\nmain EN [ 321](https://github.com/huggingface/hub-docs)\n

In [12]:
dataset[26]["answer"]

'```markdown\n# TL;DR Summary\n\nThis document provides a comprehensive guide on deploying Hugging Face Transformers models to Amazon SageMaker. Key steps include setting up the environment, deploying models trained in SageMaker or from the Hugging Face Hub, and running batch transforms. It also covers deploying large language models (LLMs) using the Hugging Face TGI container and customizing inference modules.\n```'

### Dataset Formatting

In [6]:
alpaca_promt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are a helpful assistant specialized in summarizing documents. Generate a concise TL;DR summary in markdown format having a maximum of 512 characters of the key findings from the provided documents, highlighting the most significant insights

### Input:
{}

### Response:
{}"""

### Load Model

In [ ]:
from unsloth import FastLanguageModel


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_id,
    max_seq_lengtht=4096,
    load_in_4bit=True,
)


FastLanguageModel.for_inference(model)

NotImplementedError: Unsloth: No NVIDIA GPU found? Unsloth currently only supports GPUs!

### Inference

In [ ]:
from transformers import TextStreamer

text_streamer = TextStreamer(tokenizer)

def generate_summary(
    instruction: str, streaming: bool = True, trim_input_message: bool = True,
):
    
    message = alpaca_promt.format(
            instruction,
            ""
        )
    
    inputs = tokenizer([message], return_tensors="pt").to("cuda")
    
    if streaming:
        return model.generate(**inputs, streamer=text_streamer, max_new_tokens=256, use_cache=True)
        
    else:
        output_tokens = model.generate(**inputs, max_new_tokens=250, use_cache=True,) # returns all tokens input + output
        response = tokenizer.batch_decode(output_tokens, skip_special_tokens=True)[0] 
        
        if trim_input_message:
            return response[len(message) :]
        else:
            return response


d:\Projects\SecondBrain\venv\Lib\site-packages\triton\knobs.py:212: UserWarning: Failed to find cuobjdump.exe
  warnings.warn(f"Failed to find {binary}")
d:\Projects\SecondBrain\venv\Lib\site-packages\triton\knobs.py:212: UserWarning: Failed to find nvdisasm.exe
  warnings.warn(f"Failed to find {binary}")


NameError: name 'tokenizer' is not defined

In [ ]:
_ = generate_summary(dataset[0]["instruction"], streaming=True)

In [ ]:
generate_summary(dataset[0]["instruction"], streaming=False)